## Feature Extraction

####  *AUTHOR:* Ehsan Farahbakhsh
####  *CONTACT:* e.farahbakhsh@sydney.edu.au
####  *DATE last modified:* 24/03/2026

This notebook extracts and visualises a range of kinematic features from the plate motion model and oceanic grids along trench lines within subduction zones (e.g., convergence rate and obliquity, seafloor age, sediment thickness, and carbon content).

We begin by importing the required libraries:

In [ ]:
# Import libraries
from ipywidgets import interact
import os

import cartopy.crs as ccrs
import cmcrameri.cm as ccm
import gplately
from gplately import PlateModelManager, PlateReconstruction, PlotTopologies
from gplately.tools import plate_isotherm_depth
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from scipy.interpolate import RegularGridInterpolator

# Note: Ensure the 'lib' folder is located in the same folder as this notebook
from lib.feature_extraction import *
from lib.slab_dip import calculate_slab_dip
from lib.water_thickness import calculate_water_thickness
from lib.plot import *

# Load configuration parameters (e.g., paths, model names)
from parameters import parameters

### Setup

As defined in `parameters.py`, the cell below configures the analysis parameters and specifies the paths to input/output files and directories. You can also specify the number of cores to use by setting an appropriate value for the `n_jobs` variable at the end of the cell.

**Note:** You can modify the analysis settings directly in `parameters.py`, located in the same folder as this notebook. The file is structured as a dictionary; look for keys such as `timespan` and `grid_resolution` to adjust their values as needed.

In [ ]:
# Set up temporal analysis parameters
plate_model = parameters["plate_model"]
temporal_resolution = parameters["temporal_resolution"]
time_min = parameters["timespan"]["min"] # Youngest time to analyse
time_max = parameters["timespan"]["max"] # Oldest time to analyse
# Create an array of time steps for analysis (e.g., 0, 1, 2, ... Ma)
time_steps = range(time_min, time_max + temporal_resolution, temporal_resolution)

grid_resolution = parameters["grid_resolution"]

# Directory paths for inputs and outputs
inputs_dir = parameters["inputs_dir"]
slab_h2o_dir = parameters["slab_h2o_dir"]
slab_co2_dir = parameters["slab_co2_dir"]
lookup_tables_dir = parameters["lookup_tables_dir"]
lookup_tables_dir = os.path.join(inputs_dir, lookup_tables_dir)
outputs_dir = parameters["outputs_dir"]
if not os.path.exists(outputs_dir):
    os.makedirs(outputs_dir, exist_ok=True)

subduction_data_filename = parameters["subduction_data_filename"]
subduction_data_filename = os.path.join(outputs_dir, subduction_data_filename)

# Paths to different types of oceanic grids
slab_h2o_dir = os.path.join(inputs_dir, slab_h2o_dir)
slab_co2_dir = os.path.join(inputs_dir, slab_co2_dir)
agegrid_dir = os.path.join(inputs_dir, "SeafloorAge")
spreadrate_dir = os.path.join(inputs_dir, "SpreadingRate")
sedthick_dir = os.path.join(inputs_dir, "SedimentThickness")
carbonate_dir = os.path.join(inputs_dir, "CarbonateThickness")
co2_dir = os.path.join(inputs_dir, "CrustalCO2")

lithosphere_bottom_dir = slab_h2o_dir+"Reservoirs/Lithosphere/bottom/"
lithosphere_top_dir = slab_h2o_dir+"Reservoirs/Lithosphere/top/"
crust_bound_dir = slab_h2o_dir+"Reservoirs/Crust/bound/"
sediments_bound_dir = slab_h2o_dir+"Reservoirs/Sediment/bound/"
lithosphere_carbon_dir = slab_co2_dir+"Lithosphere/"
crust_carbon_dir = slab_co2_dir+"Crust/"
serpentinite_carbon_dir = slab_co2_dir+"Serpentinite/"
sediments_carbon_dir = slab_co2_dir+"Sediment/"
organic_sediments_carbon_dir = slab_co2_dir + "Organic_Sediments/"

water_components = [lithosphere_bottom_dir,
                    crust_bound_dir,
                    sediments_bound_dir]

water_headers  = ['lithosphere_bottom',
            'crust_bound',
            'sediment_bound',
            'total']

carbon_components = [lithosphere_carbon_dir,
                    crust_carbon_dir,
                    serpentinite_carbon_dir,
                    sediments_carbon_dir,
                    organic_sediments_carbon_dir]

carbon_headers  = ['lithosphere',
            'crust',
            'serpentinite',
            'sediment',
            'organic_sediments',
            'total']

quantities = ["min", "mean", "max"]

input_water_cdf_filename  = "{}/{}/water_{}_grid_{}.nc"
input_carbon_cdf_filename  = "{}/{}/subducted_carbon_{}_{}.nc"

# Number of cores to be used for running this notebook
n_jobs = 20

In [ ]:
lookup_tables = np.empty((8, 313, 313))

for i, table in enumerate(['/Sediments/GLOSS_H2O.dat',
                           '/Metabasalts/StaudigelVolcanics_H2O.dat',
                           '/Intrusives/Intrusives_H2O.dat',
                           '/Sublithospheric_Oceanic_mantle/LOSIMAG_H2O.dat',
                           '/Sediments/GLOSS_CO2.dat',
                           '/Metabasalts/StaudigelVolcanics_CO2.dat',
                           '/Intrusives/Intrusives_CO2.dat',
                           '/Sublithospheric_Oceanic_mantle/LOSIMAG_CO2.dat']):
    H2OArray_TPZ = np.genfromtxt(lookup_tables_dir + table, skip_header=12, autostrip=True)
    lookup_tables[i] = H2OArray_TPZ[:,2].reshape(313,313)
    
T_coords = H2OArray_TPZ[:,0].reshape(313,313)[0,:]
P_coords = H2OArray_TPZ[:,1].reshape(313,313)[:,0]

lookup_interp = RegularGridInterpolator((P_coords, T_coords), lookup_tables[i],
                                        method='nearest', bounds_error=False, fill_value=None)

The cell below loads the necessary files to create the `PlateReconstruction` and `PlateTopologies` objects, which will later be used to extract features and generate visualisations.

In [ ]:
# Plate motion model
pmm = PlateModelManager()
pm = pmm.get_model(plate_model)

rotation_model = pm.get_rotation_model()
topology_features = pm.get_topologies()

static_polygons = pm.get_static_polygons()
coastlines = pm.get_coastlines()
continents = pm.get_continental_polygons()
COBs = pm.get_COBs()

plate_reconstruction = PlateReconstruction(
    rotation_model=rotation_model,
    topology_features=topology_features,
    static_polygons=static_polygons,
)

gplot = PlotTopologies(
    plate_reconstruction=plate_reconstruction,
    coastlines=coastlines,
    continents=continents,
    COBs=COBs,
)

### Feature Extraction

This cell performs the core subduction zone analysis. It either loads pre-computed subduction data or computes it from scratch using the plate motion model and oceanic grids.

If pre-computed data is available, it loads the previously saved subduction zone data from a `.csv` file. Otherwise, it loads a global plate motion model to simulate plate positions over time, determines where and how fast plates are converging, identifies subduction zones, and calculates kinematic features along trench lines within those zones. Finally, it co-registers trench locations with the corresponding oceanic grids to extract additional features.

In [ ]:
# Check if processed subduction data already exists
if os.path.isfile(subduction_data_filename):
    subduction_data = pd.read_csv(subduction_data_filename)
else:
    # Identify trench lines and extract kinematic features along them
    subduction_data = run_calculate_convergence(
        min_time=time_min,
        max_time=time_max,
        temporal_resolution=temporal_resolution,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        n_jobs=n_jobs,
        verbose=True,
    )

    # Co-register the points extracted along the trench lines with the oceanic grids
    subduction_data = run_coregister_ocean_rasters(
        times=time_steps,
        input_data=subduction_data,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        agegrid_dir=agegrid_dir,
        spreadrate_dir=spreadrate_dir,
        sedthick_dir=sedthick_dir,
        carbonate_dir=carbonate_dir,
        co2_dir=co2_dir,
        n_jobs=n_jobs,
        verbose=True,
    )

    # Calculate plate thickness
    subduction_data["plate_thickness (m)"] = plate_isotherm_depth(
        subduction_data["seafloor_age (Ma)"],
        maxiter=100,
    )

    # Calculate additional properties
    subduction_data = calculate_water_thickness(data=subduction_data)
    subduction_data = calculate_carbon(subduction_data)
    subduction_data = calculate_slab_flux(subduction_data)
    subduction_data = calculate_slab_dip(subduction_data)

    # Estimate cumulative volumes of subducted oceanic materials
    subduction_data = extract_subducted_thickness(
        subduction_data,
        plate_reconstruction=plate_reconstruction,
        rotation_model=rotation_model,
        topology_features=topology_features,
        static_polygons=static_polygons,
        grid_resolution=grid_resolution,
    )

    # Calculate fluxes of different materials into subduction zones
    subduction_data["sediment_flux (m^2/yr)"] = (
        subduction_data["sediment_thickness (m)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data["crustal_carbon_flux (t/m/yr)"] = (
        subduction_data["crustal_carbon_density (t/m^2)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data["carbonate_carbon_flux (t/m/yr)"] = (
        subduction_data["carbonate_carbon_density (t/m^2)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)    
    
    subduction_data["total_carbon_flux (t/m/yr)"] = (
        subduction_data["total_carbon_density (t/m^2)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)

    subduction_data["water_flux (m^2/yr)"] = (
        subduction_data["total_water_thickness (m)"]
        * subduction_data["convergence_rate_orthogonal (cm/yr)"] * 1.0e-2
    ).clip(0.0, np.inf)
    
    subduction_data = run_calculate_outflux(
        input_data=subduction_data,
        times=time_steps,
        water_components=water_components,
        water_headers=water_headers,
        carbon_components=carbon_components,
        carbon_headers=carbon_headers,
        lookup_tables=lookup_tables,
        lookup_interp=lookup_interp,
        agegrid_dir=agegrid_dir,
        sedthick_dir=sedthick_dir,
        lithosphere_top_dir=lithosphere_top_dir,
        input_water_cdf_filename=input_water_cdf_filename,
        input_carbon_cdf_filename=input_carbon_cdf_filename,
        n_jobs=n_jobs,
        verbose=True,
    )
    
    # Save processed data
    subduction_data.to_csv(subduction_data_filename, index=False)

Here, we identify columns or features that are either entirely NaN or constant (single unique non-NaN value), report them, and remove them from the dataset to eliminate useless features before saving the cleaned file. Then we build a list of remaining plottable features by excluding known non-feature columns and sets up a Mollweide map projection for global visualisation.

In [ ]:
nan_only = subduction_data.columns[subduction_data.isna().all()]
constant_non_nan = subduction_data.columns[
    (subduction_data.nunique(dropna=True) == 1) & (~subduction_data.isna().all())
]

if len(nan_only) > 0:
    print("NaN-only:", nan_only.tolist())

if len(constant_non_nan) > 0:
    print("Constant:", constant_non_nan.tolist())

features_to_remove = nan_only.union(constant_non_nan)
if len(features_to_remove) > 0:
    subduction_data = subduction_data.drop(columns=features_to_remove)
    subduction_data.to_csv(subduction_data_filename, index=False)

# Create list of features available for plotting
subduction_data_columns = subduction_data.columns.tolist()
features_plot = subduction_data_columns.copy()
columns_to_remove = ["lon", "lat", "age (Ma)", "subducting_plate_ID", "trench_plate_ID"]
features_plot = [x for x in features_plot if x not in columns_to_remove]

# Define map projection (Mollweide provides a good global view)
projection = ccrs.Mollweide(central_longitude=60)

### Visualisation

The cell below is interactive, allowing users to select a geological time and feature to visualise along the trench lines, with seafloor age and plate boundaries — including mid-ocean ridges and transform faults — also plotted.

In [ ]:
@interact
def show_map(time=time_steps, feature=features_plot):
    gplot.time = time

    # Load seafloor age grid for the specified geological time
    agegrid_filename =  f"seafloor_age_{time}Ma.nc"
    agegrid_file = os.path.join(agegrid_dir, agegrid_filename)    
    agegrid = gplately.grids.read_netcdf_grid(agegrid_file)

    if feature == "convergence_obliquity (degrees)":
        feat_min = -90
        feat_max = 90
    elif feature in [
        "convergence_rate (cm/yr)",
        "convergence_rate_orthogonal (cm/yr)",
        "slab_flux (m^2/yr)",
        "subduction_water_flux_lithosphere (t/m/yr)"
    ]:
        feat_min = 0
        feat_max = subduction_data[feature].quantile(0.99)
    else:
        feat_min = subduction_data[feature].quantile(0.01)
        feat_max = subduction_data[feature].quantile(0.99)
    
    subduction_data_t = subduction_data[subduction_data["age (Ma)"] == time]

    fig = plt.figure(figsize=(16, 12))
    ax = fig.add_axes(
        [0.1, 0.1, 0.8, 0.8],
        projection=projection,
        facecolor=(plt.cm.colors.to_rgba("darkgray", alpha=0.5)),
    )
    ax.set_global()

    # Create colour bar axes for the selected feature and seafloor age
    cax_feat = fig.add_axes([0.33, 0.17, 0.25, 0.02])
    cax_bg = fig.add_axes([0.62, 0.17, 0.25, 0.02])

    # Plot seafloor age as background
    bg = gplot.plot_grid(ax, agegrid.data, cmap=ccm.lapaz_r, vmin=0, vmax=230, alpha=0.7, zorder=1)
    # Plot continental areas in dark grey
    gplot.plot_coastlines(ax, facecolor="darkgray", edgecolor="none", zorder=2)
    # Plot plate motion vectors (arrows showing plate movement direction)
    gplot.plot_plate_motion_vectors(ax, spacingX=10, spacingY=10, normalise=True, alpha=0.1, zorder=3)

    # Plot feature values along trench lines
    feat = ax.scatter(subduction_data_t["lon"], subduction_data_t["lat"], 100, marker=".",
                      c=subduction_data_t[feature], cmap=ccm.hawaii_r, vmin=feat_min, vmax=feat_max,
                      transform=ccrs.PlateCarree(), zorder=4)

    gplot.plot_topological_plate_boundaries(ax, color="dimgray", linewidth=1.2, zorder=5)
    gplot.plot_trenches(ax, color="black", alpha=0.5, zorder=6)
    gplot.plot_subduction_teeth(ax, spacing=0.05, color="black", alpha=0.5, zorder=7)
    
    gl = ax.gridlines(crs=ccrs.PlateCarree(), draw_labels=True, x_inline=False, linewidth=1, color="gray", alpha=0.3, linestyle="--", zorder=8)
        
    gl.top_labels=False
    gl.bottom_labels=False
    
    gl.xlabel_style = {"size": 16}
    gl.ylabel_style = {"size": 16}

    ax.text(0.49,-0.03, "60°E", transform=ax.transAxes, fontsize=16)
    ax.text(0.46,-0.03, "0°", transform=ax.transAxes, fontsize=16)
    ax.text(0.40,-0.025, "60°W", transform=ax.transAxes, fontsize=16)

    # Create colour bars with labels
    cbar_feat = fig.colorbar(feat, cax=cax_feat, orientation="horizontal", extend="max")
    cbar_feat.set_label(format_feature_name(feature), fontsize=16, labelpad=10)
    cbar_feat.ax.tick_params(labelsize=16)

    cbar_bg = fig.colorbar(bg, cax=cax_bg, orientation="horizontal", extend="max")
    cbar_bg.set_label("Seafloor age (Myr)", fontsize=16, labelpad=10)
    cbar_bg.set_ticks([0, 50, 100, 150, 200])
    cbar_bg.ax.tick_params(labelsize=16)
    
    # Dummy handle to trigger custom handler
    trench_handle = Line2D([], [], color="black")
    
    # Add custom handles
    custom_handles = [
        Patch(facecolor="darkgray", edgecolor="none"),
        trench_handle,
        Line2D([0], [0], color="dimgray", lw=1.2),
    ]
    custom_labels = [
        "Continental crust",
        "Subduction zone",
        "Plate boundary",
    ]

    # Draw legend
    ax.legend(custom_handles, custom_labels, fontsize=16, loc="lower left", bbox_to_anchor=(0, -0.2),
              handler_map={trench_handle: HandlerTrenchLine()})
    
    ax.set_title(f"{time} Ma", fontsize=25, y=1.04)
        
    plt.show()